# 🤖 Notebook 07: Building a GPT from Scratch

We assemble a complete GPT model in MLX: token embedding + positional encoding + N transformer blocks + output projection. Then we train it on text and generate.

**Prerequisites:** Notebooks 01-06

**What you'll learn:**
1. Build a complete GPT model from transformer blocks\n2. Implement a training loop with cosine LR schedule\n3. Train on Shakespeare text and watch quality improve\n4. Generate text and see the model learn language patterns

💡 **In simple terms**, think of it as building blocks — each concept in this notebook is a building block that connects to the next. For example, understanding the basics here will make everything that follows much easier to grasp.

## ✅ Environment Validation

### 🧠 The Big Picture

Building a GPT model is like assembling a car from parts — the embedding layer is the fuel intake (converting raw text into usable form), the transformer blocks are the engine (doing the heavy processing), and the output head is the steering wheel (deciding what comes next). Training is like teaching someone to drive by showing them millions of examples.

### 💡 Why Does This Matter?

Why cosine learning rate schedule? It starts with a high learning rate for fast initial progress, then gradually reduces it for fine-grained optimization — like taking big steps when you're far from the destination and small steps when you're close. Why gradient clipping? It prevents catastrophically large updates that can destabilize training.

In [1]:
from utils.checks import validate_environment, print_environment_report
env = print_environment_report()

  Environment Validation Report
  Platform : macOS-26.4.1-arm64-arm-64bit-Mach-O
  Chip     : Apple M4 Pro
  Python   : 3.13.13  ✅
  MLX      : 0.31.1  ✅
  Metal GPU: available  ✅
  Memory   : 48.0 GB  ✅


## GPT Architecture Overview

```
Token IDs (batch, seq_len)
  │
  ▼
Token Embedding (vocab_size, d_model)
  │
  ▼
Positional Encoding (sinusoidal or RoPE)
  │
  ▼
N × TransformerBlock
  │
  ▼
Final RMSNorm
  │
  ▼
Output Projection (d_model → vocab_size)
  │
  ▼
Logits (batch, seq_len, vocab_size)
```

## 🧱 Building the GPT — One Piece at a Time

We're going to build a complete GPT model from scratch. But instead of dumping all the code at once, we'll build it **piece by piece**, testing each component before moving to the next.

Here's the plan:
1. **SwiGLU FFN** — the "thinking" layer that processes each token independently
2. **Multi-Head Attention** — the "communication" layer that lets tokens talk to each other
3. **Transformer Block** — combines attention + FFN into one reusable unit
4. **GPTModel** — stacks multiple blocks together with embeddings and an output head

💡 **Think of it like LEGO**: each piece is simple on its own. The magic comes from stacking them together.

### Step 1: The SwiGLU Feed-Forward Network

Each token passes through this network independently. It's like a "thinking" step — the token processes its own information without looking at other tokens.

**SwiGLU** uses a gating mechanism: one path computes "what to say" (`w1`), another computes "how much to say" (`w_gate`), and they're multiplied together. This lets the network learn to selectively amplify or suppress features.

```
Input x (d_model) → [gate path: SiLU(W_gate · x)] × [value path: W1 · x] → W2 → Output (d_model)
```

💡 **Why SwiGLU instead of plain ReLU?** The gating mechanism gives the network more control over information flow. It's used in LLaMA, Gemma, Mistral, and most modern LLMs.

In [2]:
import mlx.core as mx
import mlx.nn as nn
import math
import numpy as np

class SwiGLUFFN(nn.Module):
    """Feed-forward network with SwiGLU gating — the 'thinking' layer.
    
    Each token is processed independently (no cross-token communication).
    Input shape:  (batch, seq, d_model)
    Output shape: (batch, seq, d_model)
    """
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)      # Value path
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)   # Gate path
        self.w2 = nn.Linear(d_ff, d_model, bias=False)       # Project back down
    
    def __call__(self, x):
        # Gate: SiLU activation decides "how much" of each feature to keep
        gate = nn.silu(self.w_gate(x))   # shape: (batch, seq, d_ff)
        # Value: raw features
        value = self.w1(x)               # shape: (batch, seq, d_ff)
        # Multiply gate × value, then project back to d_model
        return self.w2(gate * value)     # shape: (batch, seq, d_model)

# Quick test — verify shapes are preserved
test_ffn = SwiGLUFFN(d_model=64, d_ff=256)
test_input = mx.random.normal((2, 8, 64))  # (batch=2, seq=8, d_model=64)
test_output = test_ffn(test_input)
mx.eval(test_output)
print(f"SwiGLU FFN: {test_input.shape} → {test_output.shape} ✅")
print(f"Parameters: gate={64*256:,} + value={64*256:,} + down={256*64:,} = {3*64*256:,} total")

SwiGLU FFN: (2, 8, 64) → (2, 8, 64) ✅
Parameters: gate=16,384 + value=16,384 + down=16,384 = 49,152 total


### Step 2: Multi-Head Attention — Letting Tokens Talk

This is the core innovation of the Transformer. Each token can "look at" every previous token and decide what information to gather.

**The analogy**: Imagine you're at a party (the sequence). Each person (token) wants to have conversations. Multi-head attention is like having multiple simultaneous conversations — one head might focus on grammar, another on meaning, another on position.

**The math in plain English**:
1. Each token creates three vectors: **Query** ("what am I looking for?"), **Key** ("what do I have to offer?"), **Value** ("here's my actual information")
2. Each Query is compared against all Keys to get attention scores ("how relevant is each token to me?")
3. Scores are passed through softmax to become probabilities (sum to 1.0)
4. The probabilities weight the Values — tokens with high scores contribute more

**Causal masking**: In GPT, token at position 5 can only see tokens 0-5 (not future tokens). We enforce this with a triangular mask.

In [3]:
class MultiHeadAttention(nn.Module):
    """Multi-head self-attention — the 'communication' layer.
    
    Lets each token gather information from all previous tokens.
    Input shape:  (batch, seq, d_model)
    Output shape: (batch, seq, d_model)
    """
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads  # Each head gets a slice of the model dimension
        self.scale = math.sqrt(self.d_head)  # Scaling factor for stable softmax
        
        # Q, K, V projections — each creates a different "view" of the input
        self.W_q = nn.Linear(d_model, d_model, bias=False)  # "What am I looking for?"
        self.W_k = nn.Linear(d_model, d_model, bias=False)  # "What do I have to offer?"
        self.W_v = nn.Linear(d_model, d_model, bias=False)  # "Here's my actual info"
        self.W_o = nn.Linear(d_model, d_model, bias=False)  # Combine all heads
    
    def __call__(self, x, mask=None):
        B, T, D = x.shape  # batch, sequence length, model dimension
        
        # Step 1: Project to Q, K, V
        Q = self.W_q(x)  # (B, T, d_model)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # Step 2: Split into multiple heads
        # Reshape: (B, T, d_model) → (B, T, n_heads, d_head) → (B, n_heads, T, d_head)
        Q = Q.reshape(B, T, self.n_heads, self.d_head).transpose(0, 2, 1, 3)
        K = K.reshape(B, T, self.n_heads, self.d_head).transpose(0, 2, 1, 3)
        V = V.reshape(B, T, self.n_heads, self.d_head).transpose(0, 2, 1, 3)
        
        # Step 3: Compute attention scores — "how relevant is each token to me?"
        scores = (Q @ K.transpose(0, 1, 3, 2)) / self.scale  # (B, n_heads, T, T)
        
        # Step 4: Apply causal mask — can't look at future tokens!
        if mask is not None:
            scores = mx.where(mask == 0, mx.array(float('-inf')), scores)
        
        # Step 5: Softmax → attention weights (probabilities that sum to 1)
        weights = mx.softmax(scores, axis=-1)  # (B, n_heads, T, T)
        
        # Step 6: Weighted sum of values
        out = (weights @ V).transpose(0, 2, 1, 3).reshape(B, T, D)  # (B, T, d_model)
        
        # Step 7: Final projection to combine all heads
        return self.W_o(out)

# Quick test
test_attn = MultiHeadAttention(d_model=64, n_heads=4)
mask = mx.tril(mx.ones((8, 8)))  # Causal mask: lower triangular
test_out = test_attn(test_input, mask)
mx.eval(test_out)
print(f"Multi-Head Attention: {test_input.shape} → {test_out.shape} ✅")
print(f"  {4} heads × {16} dims per head = {64} total (d_model)")
print(f"  Causal mask shape: {mask.shape} (lower triangular — no peeking at future!)")

Multi-Head Attention: (2, 8, 64) → (2, 8, 64) ✅
  4 heads × 16 dims per head = 64 total (d_model)
  Causal mask shape: (8, 8) (lower triangular — no peeking at future!)


### Step 3: The Transformer Block — Attention + FFN + Residuals

A Transformer block combines attention (communication) and FFN (thinking) with two critical additions:
- **RMSNorm** before each sub-layer — keeps numbers in a stable range
- **Residual connections** (the `x = x + ...` pattern) — lets gradients flow easily during training

```
Input x
  │
  ├──→ RMSNorm → Attention → + ──→ (x + attention output)
  │                           ↑
  └───────────────────────────┘  ← residual connection
  │
  ├──→ RMSNorm → FFN → + ──→ (x + FFN output)  
  │                      ↑
  └──────────────────────┘  ← residual connection
  │
Output
```

💡 **Why residual connections?** Without them, gradients vanish in deep networks (the signal gets weaker with each layer). Residuals create a "highway" that lets the gradient flow directly from output to input.

In [4]:
class TransformerBlock(nn.Module):
    """One Transformer block: Attention + FFN with pre-norm and residuals.
    
    Input shape:  (batch, seq, d_model)
    Output shape: (batch, seq, d_model)
    """
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn_norm = nn.RMSNorm(d_model)   # Normalize before attention
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ffn_norm = nn.RMSNorm(d_model)    # Normalize before FFN
        self.ffn = SwiGLUFFN(d_model, d_ff)
    
    def __call__(self, x, mask=None):
        # Communication: tokens talk to each other via attention
        x = x + self.attn(self.attn_norm(x), mask)   # Residual connection!
        # Thinking: each token processes its own information
        x = x + self.ffn(self.ffn_norm(x))            # Residual connection!
        return x

# Quick test
test_block = TransformerBlock(d_model=64, n_heads=4, d_ff=256)
test_out = test_block(test_input, mask)
mx.eval(test_out)
print(f"Transformer Block: {test_input.shape} → {test_out.shape} ✅")
print(f"  Components: RMSNorm → Attention → RMSNorm → SwiGLU FFN")

Transformer Block: (2, 8, 64) → (2, 8, 64) ✅
  Components: RMSNorm → Attention → RMSNorm → SwiGLU FFN


### Step 4: The Complete GPT Model

Now we stack everything together:
1. **Token Embedding** — converts token IDs (integers) to vectors
2. **Position Embedding** — tells the model WHERE each token is in the sequence
3. **N × Transformer Blocks** — the repeated attention + FFN layers
4. **Final Norm + Output Projection** — converts vectors back to vocabulary probabilities

The `generate` method does autoregressive text generation: predict one token, append it, repeat.

### Putting It All Together: GPTModel

### 📦 Additional Imports

The next cell adds the remaining imports we need for training.

In [5]:
import mlx.optimizers as optim
import time
import matplotlib.pyplot as plt

class GPTModel(nn.Module):
    """Complete GPT model: Embeddings → N × TransformerBlock → Output Head.
    
    This is the same architecture used by GPT-2, GPT-3, and LLaMA (with minor variations).
    
    Forward pass:
        token_ids (B, T) → embeddings → transformer blocks → logits (B, T, vocab_size)
    
    Generate:
        Autoregressive: predict one token at a time, append, repeat.
    """
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, max_seq_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)     # Token → vector
        self.pos_emb = nn.Embedding(max_seq_len, d_model)      # Position → vector
        self.blocks = [TransformerBlock(d_model, n_heads, d_ff) # N transformer blocks
                       for _ in range(n_layers)]
        self.final_norm = nn.RMSNorm(d_model)                  # Final normalization
        self.output_proj = nn.Linear(d_model, vocab_size, bias=False)  # Vector → vocab probs
        self.max_seq_len = max_seq_len
    
    def __call__(self, token_ids):
        """Forward pass: token IDs → logits over vocabulary."""
        B, T = token_ids.shape
        
        # Step 1: Convert token IDs to vectors + add position info
        tok_emb = self.token_emb(token_ids)   # (B, T, d_model)
        pos_emb = self.pos_emb(mx.arange(T))  # (T, d_model)
        x = tok_emb + pos_emb                 # (B, T, d_model) — broadcasting adds position
        
        # Step 2: Create causal mask (lower triangular — can't see future)
        mask = mx.tril(mx.ones((T, T)))        # (T, T)
        
        # Step 3: Pass through all transformer blocks
        for block in self.blocks:
            x = block(x, mask)                 # (B, T, d_model)
        
        # Step 4: Final norm + project to vocabulary
        x = self.final_norm(x)                 # (B, T, d_model)
        return self.output_proj(x)             # (B, T, vocab_size)
    
    def generate(self, prompt_ids, max_tokens=50, temperature=1.0):
        """Generate text autoregressively: predict one token, append, repeat."""
        tokens = list(prompt_ids.tolist()) if hasattr(prompt_ids, 'tolist') else list(prompt_ids)
        for _ in range(max_tokens):
            ctx = tokens[-self.max_seq_len:]       # Use last max_seq_len tokens as context
            x = mx.array([ctx])                    # (1, context_len)
            logits = self(x)                       # (1, context_len, vocab_size)
            logits = logits[0, -1, :] / temperature  # Last position, scaled by temperature
            probs = mx.softmax(logits, axis=-1)
            mx.eval(probs)
            probs_np = np.array(probs, dtype=np.float64)
            probs_np /= probs_np.sum()
            next_token = int(np.random.choice(len(probs_np), p=probs_np))
            tokens.append(next_token)
        return tokens

# Test the complete model
test_model = GPTModel(vocab_size=256, d_model=64, n_heads=4, n_layers=2, d_ff=256, max_seq_len=128)
mx.eval(test_model.parameters())

# Count parameters
total_params = sum(p.size for _, p in nn.utils.tree_flatten(test_model.parameters()))
print(f"GPTModel created ✅")
print(f"  Config: vocab=256, d_model=64, heads=4, layers=2, d_ff=256")
print(f"  Total parameters: {total_params:,}")
print(f"  Memory (float32): {total_params * 4 / 1024:.1f} KB")

# Quick forward pass test
test_ids = mx.array([[1, 5, 12, 8, 3, 42, 7, 99]])  # (1, 8)
test_logits = test_model(test_ids)
mx.eval(test_logits)
print(f"\n  Forward pass: {test_ids.shape} → {test_logits.shape}")
print(f"  ✅ Input (1, 8) → Output (1, 8, 256) — one probability distribution per position")

GPTModel created ✅
  Config: vocab=256, d_model=64, heads=4, layers=2, d_ff=256
  Total parameters: 172,352
  Memory (float32): 673.2 KB

  Forward pass: (1, 8) → (1, 8, 256)
  ✅ Input (1, 8) → Output (1, 8, 256) — one probability distribution per position


### 🔍 What Just Happened?

We just trained a language model from scratch! The model learned to predict the next character by processing sequences through embedding → transformer blocks → output projection. The training loop pattern (forward → loss → backward → clip → update → eval) is the same pattern used to train GPT-4, just at a much smaller scale.

## Model Inspection

Let's count parameters and estimate memory for different configs.

In [6]:
configs = {
    'tiny':   dict(vocab_size=256, d_model=64,  n_heads=4, n_layers=2, d_ff=256,  max_seq_len=128),
    'small':  dict(vocab_size=256, d_model=128, n_heads=4, n_layers=4, d_ff=512,  max_seq_len=256),
    'medium': dict(vocab_size=256, d_model=256, n_heads=8, n_layers=6, d_ff=1024, max_seq_len=256),
}

import mlx.utils as _mxu
for name, cfg in configs.items():
    model = GPTModel(**cfg)
    # Simple param count
    n_params = sum(p.size for _, p in _mxu.tree_flatten(model.parameters()))
    mem_mb = n_params * 4 / 1e6  # float32
    print(f'{name:>7s}: {n_params:>10,} params ({mem_mb:.1f} MB float32)')

   tiny:    172,352 params (0.7 MB float32)
  small:  1,148,032 params (4.6 MB float32)
 medium:  6,491,392 params (26.0 MB float32)


### ⚠️ Handling Common Errors

When working with ML code, errors are normal and expected. Here's a pattern for handling them gracefully — if something goes wrong, you get a helpful message instead of a crash.

In [7]:
# 💡 Error handling pattern — use this when operations might fail
try:
    # This is where your ML code goes
    import mlx.core as mx
    test = mx.array([1.0, 2.0, 3.0])
    result = mx.sum(test)
    mx.eval(result)
    print(f'✅ Success! Result: {result.item()}')
except Exception as e:
    print(f'❌ Error: {e}')
    print('💡 Tip: Check that MLX is installed and your inputs are valid')

✅ Success! Result: 6.0


## Data Preparation

We'll use a simple text dataset with character-level tokenization for training our tiny GPT.

In [8]:
# Download tiny shakespeare or use inline text
text = """To be, or not to be, that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles,
And by opposing end them. To die, to sleep;
No more; and by a sleep to say we end
The heart-ache and the thousand natural shocks
That flesh is heir to, 'tis a consummation
Devoutly to be wish'd. To die, to sleep;
To sleep: perchance to dream: ay, there's the rub;
For in that sleep of death what dreams may come
When we have shuffled off this mortal coil."""

# Character-level tokenizer
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join(itos[i] for i in ids)

data = mx.array(encode(text))
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f'Vocab size: {vocab_size}')
print(f'Train tokens: {len(train_data)}, Val tokens: {len(val_data)}')
print(f'Sample: "{text[:60]}..."')
print(f'Encoded: {encode(text[:20])}')

Vocab size: 38
Train tokens: 475, Val tokens: 53
Sample: "To be, or not to be, that is the question:
Whether 'tis nobl..."
Encoded: [13, 28, 1, 16, 19, 3, 1, 28, 31, 1, 27, 28, 33, 1, 33, 28, 1, 16, 19, 3]


## Training Loop

Standard MLX training: `nn.value_and_grad` → `optimizer.update` → `mx.eval`.

In [9]:
# Config
seq_len = 64
batch_size = 8
lr = 3e-4
n_steps = 300

model = GPTModel(vocab_size=vocab_size, d_model=64, n_heads=4, n_layers=2, d_ff=256, max_seq_len=seq_len)
optimizer = optim.AdamW(learning_rate=lr, weight_decay=0.1)

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = mx.random.randint(0, len(d) - seq_len, shape=(batch_size,))
    mx.eval(ix)
    ix = ix.tolist()
    x = mx.stack([d[i:i+seq_len] for i in ix])
    y = mx.stack([d[i+1:i+seq_len+1] for i in ix])
    return x, y

def loss_fn(model, x, y):
    logits = model(x)  # (B, T, vocab_size)
    return nn.losses.cross_entropy(logits, y, reduction='mean')

loss_and_grad_fn = nn.value_and_grad(model, loss_fn)

losses = []
t0 = time.perf_counter()
for step in range(n_steps):
    x, y = get_batch('train')
    loss, grads = loss_and_grad_fn(model, x, y)
    optimizer.update(model, grads)
    mx.eval(model.parameters(), optimizer.state, loss)
    loss_val = loss.item()
    losses.append(loss_val)
    if step % 50 == 0 or step == n_steps - 1:
        print(f'Step {step:4d} | Loss: {loss_val:.4f} | PPL: {math.exp(loss_val):.1f}')

t1 = time.perf_counter()
print(f'\nTraining done in {t1-t0:.1f}s')

Step    0 | Loss: 3.8212 | PPL: 45.7


Step   50 | Loss: 1.7505 | PPL: 5.8


Step  100 | Loss: 1.0940 | PPL: 3.0


Step  150 | Loss: 0.6391 | PPL: 1.9


Step  200 | Loss: 0.3727 | PPL: 1.5


Step  250 | Loss: 0.2624 | PPL: 1.3
Step  299 | Loss: 0.1599 | PPL: 1.2

Training done in 0.7s


## Loss Curve & Text Generation

In [10]:
from utils.viz import plot_loss_curve
fig = plot_loss_curve(losses, title='GPT Training Loss')
plt.show()

# Generate text
prompt = 'To be'
prompt_ids = encode(prompt)
generated = model.generate(prompt_ids, max_tokens=100, temperature=0.8)
print(f'Prompt: "{prompt}"')
print(f'Generated: "{decode(generated)}"')

Prompt: "To be"
Generated: "To be d
po dre, by to sleep p;
No more; and a slep by ay say che t-ay he he nd
The t-andrt-ache the thoch"


/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_24984/1699775953.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---

### 🎯 Interview Question nb07-q01  ·  [warmup]  ·  mle, research_engineer

**Q:** Derive the softmax-temperature formula `softmax(logits / T)`. What do T→0 and T→∞ limits give you, and what's the typical T range for (a) creative writing, (b) code generation, (c) deterministic extraction?

<details>
<summary>Key points in a strong answer</summary>

- Starting point: `softmax(x)_i = exp(x_i) / Σ_j exp(x_j)`. With temperature T > 0: `softmax(x / T)_i = exp(x_i / T) / Σ_j exp(x_j / T)`.
- T → 0⁺: exponent ratios blow up; the largest logit dominates and `softmax` → one-hot at argmax. Equivalent to greedy decoding.
- T → ∞: `x / T → 0`, `exp(0) = 1`, so the distribution → uniform over the vocabulary. Effectively random sampling.
- T = 1: the model's native distribution — what it was trained to emit.
- Typical production ranges: creative writing T ∈ [0.7, 1.0] (variety without chaos); code generation T ∈ [0.0, 0.4] (deterministic / low-variance so syntax stays valid); extraction / structured output T = 0 or ≤ 0.1 (greedy is OK because one right answer).
- Interaction with top-p / top-k: temperature scales the logits BEFORE truncation. High T + tight top-p can produce the model's "creative but still plausible" behavior — the hallmark of ChatGPT default (T=1, top_p=1.0 for the free tier; T=0.8, top_p=0.95 for tuned deployments).
- Numerically stable implementation: max-subtract BEFORE dividing by T to prevent bf16/fp16 overflow on large logits. Most frameworks do this automatically inside `softmax`, but hand-rolled implementations forget it.
</details>

> ⚠️ **Trap:** Saying "T=0 is random". Quite the opposite — T→0 is fully deterministic (greedy). T→∞ is random. Beginners reverse this because "zero means nothing" feels like "no information".
>
> 📚 **References:** https://platform.openai.com/docs/guides/text-generation, https://arxiv.org/abs/1904.09751

---

### 🎯 Interview Question nb07-q02  ·  [core]  ·  mle, research_engineer, systems_engineer

**Q:** Compare greedy, top-k, top-p (nucleus), and beam search decoding. Write the math for each and explain when each wins and loses.

<details>
<summary>Key points in a strong answer</summary>

- **Greedy**: `next_token = argmax(logits)`. O(1) per step. Deterministic but degenerates on open-ended generation (loops, low diversity). Wins on structured output (JSON extraction, code, math).
- **Top-k** (Fan et al. 2018): keep only the top-k logits, zero the rest, renormalize, sample. Controls output vocabulary cardinality. Fails when the "right" distribution is FLAT (many equally likely tokens) — forces a choice among a fixed-cardinality subset regardless of the distribution's natural shape.
- **Top-p / Nucleus** (Holtzman et al. 2019): sort logits descending; find the smallest subset whose cumulative probability ≥ p; sample from that nucleus after renormalization. Adapts cardinality to distribution sharpness — narrow when one token dominates, wide when many compete. The 2024 default for chat.
- **Beam search**: maintain B parallel hypotheses; at each step, expand each to B continuations; keep the top-B by cumulative log-prob. Finds more globally-optimal sequences than greedy. Degenerates to N-gram-repetitive output on open-ended generation (the famous "beam search curse" — high likelihood ≠ high quality). Still standard for TRANSLATION and SUMMARIZATION where target is well-defined.
- **Math ties**: greedy ≈ top-1 ≈ beam with B=1 ≈ argmax. Top-p with p=1.0 ≈ temperature sampling. Top-k with k=|V| ≈ no filtering.
- **Production defaults (2024)**: chat T ∈ [0.7, 0.8], top_p ∈ [0.9, 0.95], top_k=0 (off); code T ∈ [0.1, 0.3], top_p=0.95; beam for translation (B=4-8) only.
- **When they fail on short context**: small models at T=0 loop ("the the the"); large models at T=0 with code can produce repeated code blocks. Top-p is the usual fix — it introduces just enough randomness to break determinism without harming coherence.
</details>

> ⚠️ **Trap:** Claiming "beam search always finds the best output". It finds the HIGHEST-LIKELIHOOD sequence under the model's distribution, but that often correlates NEGATIVELY with human quality on open-ended tasks (repetition, overly-confident generic text). This is a well-documented failure mode — the model's likelihood is a BAD proxy for quality outside constrained tasks.
>
> 📚 **References:** https://arxiv.org/abs/1805.04833, https://arxiv.org/abs/1904.09751, https://arxiv.org/abs/2004.10450

---

### 🧑‍💻 Whiteboard Challenge: Implement top-p (nucleus) sampling from logits

**Prompt:** Implement `top_p_sample(logits, p, seed)` that returns a single sampled token index using nucleus sampling. Assert (a) the sampled token's CUMULATIVE probability rank is ≤ the nucleus boundary, (b) the renormalized nucleus sums to 1.0 within 1e-5, and (c) at p=1.0 the implementation reduces to plain temperature-1.0 sampling.

**Constraints:**
- Pure MLX; `logits` is a 1D `mx.array` of shape `(V,)`. Return a Python `int` for the sampled token id.
- Sort logits descending via `mx.argsort`, compute `softmax` and `cumsum`, find the smallest index `k` where `cumsum[k] >= p`, zero the tail, renormalize, sample.
- Use `mx.random.categorical` or manual inverse-CDF sampling via `mx.random.uniform`. Use `mx.eval` before any `.item()` call.
- Assert the sampled index is within the nucleus set (by verifying `probs_renormalized[sampled] > 0`).
- Assert the renormalized probs over the nucleus sum to ≈ 1.0 (tolerance 1e-5).

**Expected complexity:** O(V log V) from the sort (dominant); O(V) for cumsum, masking, sampling. Negligible vs the attention cost per step.

In [11]:
import mlx.core as mx


def top_p_sample(logits: mx.array, p: float, seed: int = 0) -> int:
    """Nucleus (top-p) sampling. `logits` is 1D of shape (V,). Returns a Python int."""
    assert logits.ndim == 1, f"expected 1-D logits, got shape {logits.shape}"
    assert 0.0 < p <= 1.0, f"p must be in (0, 1], got {p}"

    # Sort descending by logit.
    _order = mx.argsort(-logits)          # (V,), indices sorted high-to-low
    _sorted_logits = logits[_order]
    _sorted_probs = mx.softmax(_sorted_logits, axis=-1)
    _cum = mx.cumsum(_sorted_probs, axis=-1)
    mx.eval(_sorted_probs, _cum)

    # Find smallest k such that cumsum[k] >= p — that's the nucleus boundary.
    _mask = (_cum >= p).astype(mx.int32)
    # argmax of (>=p) gives the first True position.
    _k = int(mx.argmax(_mask).item())
    # Keep indices 0..k (inclusive), zero the rest, renormalize.
    _keep = mx.arange(_sorted_probs.shape[0]) <= _k
    _nucleus_probs = mx.where(_keep, _sorted_probs, mx.zeros_like(_sorted_probs))
    _renorm = _nucleus_probs / mx.sum(_nucleus_probs)
    mx.eval(_renorm)

    # Sample via MLX's categorical API from the log-probs of the renormalized dist.
    mx.random.seed(seed)
    _sampled_sorted_idx = int(mx.random.categorical(mx.log(_renorm + 1e-30)).item())
    # Map sorted index back to original vocab index.
    return int(_order[_sampled_sorted_idx].item())


# -- Test 1: peaked distribution (p=0.9) should pick from the top few. ---------
mx.random.seed(0)
_V = 50
_logits = mx.concatenate([
    mx.array([5.0, 4.5, 3.5, 2.0]),          # top-4 are clearly peaked
    mx.random.normal(shape=(_V - 4,)) * 0.5,  # rest are noise near 0
])
mx.eval(_logits)

_sampled = top_p_sample(_logits, p=0.9, seed=42)
# The sampled token must come from the nucleus — verify its cumulative-prob rank.
_probs = mx.softmax(_logits, axis=-1)
_order = mx.argsort(-_logits)
_sorted_probs = _probs[_order]
_cum = mx.cumsum(_sorted_probs, axis=-1)
mx.eval(_probs, _sorted_probs, _cum)
# Find the nucleus size k.
_k = int(mx.argmax((_cum >= 0.9).astype(mx.int32)).item())
_nucleus_set = set(int(_order[i].item()) for i in range(_k + 1))
assert _sampled in _nucleus_set, f"sampled token {_sampled} not in nucleus {_nucleus_set}"
print(f"[1] peaked dist (p=0.9): sampled token = {_sampled}, nucleus size = {_k + 1}")

# -- Test 2: renormalized nucleus sums to 1.0 within 1e-5. ---------------------
_mask = mx.arange(_V) <= _k
_nucleus_probs = mx.where(_mask, _sorted_probs, mx.zeros_like(_sorted_probs))
_renorm = _nucleus_probs / mx.sum(_nucleus_probs)
mx.eval(_renorm)
_sum = float(mx.sum(_renorm).item())
assert abs(_sum - 1.0) < 1e-5, f"renormalized nucleus sums to {_sum:.6f}, expected 1.0"
print(f"[2] renormalized nucleus sum = {_sum:.6f}")

# -- Test 3: at p=1.0, top-p reduces to plain temperature-1 sampling. ---------
# We test by Monte-Carlo: 1000 samples at p=1.0 should spread across the vocab,
# and the most-frequent token should be the argmax (with tight frequency pinned above 1/V).
_counts = {i: 0 for i in range(_V)}
for _seed in range(1000):
    _tok = top_p_sample(_logits, p=1.0, seed=_seed)
    _counts[_tok] = _counts.get(_tok, 0) + 1

_argmax = int(mx.argmax(_logits).item())
# Argmax token should appear roughly with its softmax probability (≈ 0.55).
_argmax_freq = _counts[_argmax] / 1000
_argmax_prob = float(_probs[_argmax].item())
print(f"[3] p=1.0 Monte-Carlo: argmax={_argmax} freq={_argmax_freq:.3f} vs prob={_argmax_prob:.3f}")
# Tolerance is generous because 1000 samples is noisy.
assert abs(_argmax_freq - _argmax_prob) < 0.1, (
    f"at p=1.0 sampling should match distribution; got freq={_argmax_freq:.3f} vs prob={_argmax_prob:.3f}"
)

print("\n✅ Top-p sampling: nucleus constraint, renormalization, and p=1.0 reduction all verified.")

[1] peaked dist (p=0.9): sampled token = 1, nucleus size = 14
[2] renormalized nucleus sum = 1.000000


[3] p=1.0 Monte-Carlo: argmax=0 freq=0.442 vs prob=0.441

✅ Top-p sampling: nucleus constraint, renormalization, and p=1.0 reduction all verified.


## 🔧 Deep Rework: Full Training Pipeline

The sections below upgrade this notebook from a toy demo to a proper training pipeline with:
- 📦 Real dataset (tiny_shakespeare) with train/val split
- 📈 Cosine LR schedule with linear warmup
- ✂️ Gradient clipping
- 💾 Checkpoint save/load with NaN recovery
- 📊 Evaluation loop (validation loss + perplexity)
- ✍️ Generation samples showing quality progression

All heavy lifting lives in `utils/training.py` — notebook cells just import and call.

## 📦 Task 9.1 — Dataset Download & Preparation

💡 We use the **tiny_shakespeare** dataset (~1 MB) — all of Shakespeare's works concatenated. The `download_tiny_shakespeare` function fetches it once and caches locally, then splits 90/10 into train/val.

🎯 **Interview tip:** Real training always needs a held-out validation set to detect overfitting.

In [12]:
from utils.training import download_tiny_shakespeare, CharTokenizer

# Download and split dataset
train_text, val_text = download_tiny_shakespeare(data_dir="data", val_fraction=0.1)
print(f"Train chars: {len(train_text):,}  |  Val chars: {len(val_text):,}")
print(f"Sample: {train_text[:120]}…")

# Build character tokenizer
tokenizer = CharTokenizer(train_text + val_text)
print(f"\n💡 Vocab size: {tokenizer.vocab_size}")
print(f"Encode 'Hello': {tokenizer.encode('Hello')}")
print(f"Decode back:    {tokenizer.decode(tokenizer.encode('Hello'))}")

# Tokenize into MLX arrays
train_ids = mx.array(tokenizer.encode(train_text))
val_ids = mx.array(tokenizer.encode(val_text))
print(f"\nTrain tokens: {train_ids.shape[0]:,}  |  Val tokens: {val_ids.shape[0]:,}")

✅ Found cached data/tiny_shakespeare.txt
Train chars: 1,003,854  |  Val chars: 111,540
Sample: First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved ra…

💡 Vocab size: 65
Encode 'Hello': [20, 43, 50, 50, 53]
Decode back:    Hello

Train tokens: 1,003,854  |  Val tokens: 111,540


## 📈 Task 9.2 — Full Training Pipeline

⚡ **Cosine LR schedule with linear warmup** — the standard recipe used by GPT-3, LLaMA, and most modern LLMs:

$$\text{lr}(t) = \begin{cases} \text{max\_lr} \times \frac{t}{\text{warmup}} & t < \text{warmup} \\ \text{min\_lr} + \frac{1}{2}(\text{max\_lr} - \text{min\_lr})\left(1 + \cos\frac{\pi \cdot t}{\text{max\_steps}}\right) & t \geq \text{warmup} \end{cases}$$

💡 **Why warmup?** Starting with a high learning rate is like trying to park a car at full speed — the model's randomly initialized weights produce wild gradients early on, and a large LR amplifies that chaos, often causing divergence or NaN losses. Warmup gradually increases the rate over the first few steps so the optimizer can "feel out" the loss landscape before taking big steps. Once the gradients stabilize, we ramp up to the full learning rate and then cosine-decay it back down.

⚠️ **Gradient clipping** prevents a single bad batch from blowing up training. We clip the global norm to `max_norm`.

In [13]:
from utils.training import cosine_lr_schedule, clip_grad_norm

# --- Visualize the LR schedule ---
max_lr, min_lr = 3e-4, 1e-5
warmup_steps, max_steps = 20, 200
steps = list(range(max_steps))
lrs = [cosine_lr_schedule(s, warmup_steps, max_steps, max_lr, min_lr) for s in steps]

plt.figure(figsize=(8, 3))
plt.plot(steps, lrs, linewidth=2)
plt.axvline(warmup_steps, color='red', linestyle='--', alpha=0.5, label='warmup end')
plt.xlabel('Step'); plt.ylabel('Learning Rate')
plt.title('💡 Cosine LR Schedule with Linear Warmup')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.show()
print(f"LR at step 0: {lrs[0]:.2e}  |  peak: {max(lrs):.2e}  |  final: {lrs[-1]:.2e}")

LR at step 0: 0.00e+00  |  peak: 2.93e-04  |  final: 1.00e-05


/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_24984/4026532063.py:14: UserWarning: Glyph 128161 (\N{ELECTRIC LIGHT BULB}) missing from font(s) DejaVu Sans.
  plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_24984/4026532063.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The next cell continues the implementation:

**--- Full training loop with cosine LR + gradient clipping ---**

In [14]:
# --- Full training loop with cosine LR + gradient clipping ---
seq_len = 64
batch_size = 8
max_lr = 3e-4
min_lr = 1e-5
warmup_steps = 20
n_steps = 200
clip_norm = 1.0

# Fresh model on the real dataset
model = GPTModel(
    vocab_size=tokenizer.vocab_size, d_model=64, n_heads=4,
    n_layers=2, d_ff=256, max_seq_len=seq_len,
)
optimizer = optim.AdamW(learning_rate=max_lr, weight_decay=0.1)

def get_batch(data, batch_size, seq_len):
    ix = np.random.randint(0, len(data) - seq_len, size=(batch_size,))
    x = mx.stack([data[int(i):int(i)+seq_len] for i in ix])
    y = mx.stack([data[int(i)+1:int(i)+seq_len+1] for i in ix])
    return x, y

def loss_fn(model, x, y):
    logits = model(x)
    return nn.losses.cross_entropy(logits, y, reduction='mean')

loss_and_grad_fn = nn.value_and_grad(model, loss_fn)

train_losses, grad_norms, lr_history = [], [], []
t0 = time.perf_counter()

for step in range(n_steps):
    # Cosine LR schedule
    lr = cosine_lr_schedule(step, warmup_steps, n_steps, max_lr, min_lr)
    optimizer.learning_rate = lr
    lr_history.append(lr)

    x, y = get_batch(train_ids, batch_size, seq_len)
    loss, grads = loss_and_grad_fn(model, x, y)

    # Gradient clipping
    grads, gnorm = clip_grad_norm(grads, clip_norm)
    grad_norms.append(gnorm)

    optimizer.update(model, grads)
    mx.eval(model.parameters(), optimizer.state, loss)

    loss_val = loss.item()
    train_losses.append(loss_val)

    if step % 50 == 0 or step == n_steps - 1:
        print(f"Step {step:4d} | Loss: {loss_val:.4f} | PPL: {math.exp(loss_val):.1f} | "
              f"LR: {lr:.2e} | GradNorm: {gnorm:.3f}")

elapsed = time.perf_counter() - t0
print(f"\n⚡ Training done in {elapsed:.1f}s ({n_steps/elapsed:.0f} steps/s)")

Step    0 | Loss: 4.3299 | PPL: 75.9 | LR: 0.00e+00 | GradNorm: 1.954
Step   50 | Loss: 2.7720 | PPL: 16.0 | LR: 2.58e-04 | GradNorm: 0.629


Step  100 | Loss: 2.6233 | PPL: 13.8 | LR: 1.55e-04 | GradNorm: 0.601
Step  150 | Loss: 2.5706 | PPL: 13.1 | LR: 5.25e-05 | GradNorm: 0.589


Step  199 | Loss: 2.5545 | PPL: 12.9 | LR: 1.00e-05 | GradNorm: 0.571

⚡ Training done in 0.5s (415 steps/s)


## 💾 Task 9.4 — Checkpoint Save/Load & NaN Recovery

💡 `CheckpointManager` persists model weights, optimizer state, and step counter to disk. If training hits NaN, we reload the last good checkpoint and reduce LR.

⚠️ **Pitfall:** Without checkpointing, a NaN at step 900 of 1000 means starting over from scratch.

In [15]:
from utils.training import CheckpointManager

ckpt_mgr = CheckpointManager(checkpoint_dir="checkpoints/gpt_demo")

# Save current training state
ckpt_path = ckpt_mgr.save(model, optimizer, step=n_steps)
print(f"💾 Saved checkpoint to: {ckpt_path}")

# Demonstrate round-trip: load into a fresh model
model_restored = GPTModel(
    vocab_size=tokenizer.vocab_size, d_model=64, n_heads=4,
    n_layers=2, d_ff=256, max_seq_len=seq_len,
)
optimizer_restored = optim.AdamW(learning_rate=max_lr, weight_decay=0.1)
restored_step = ckpt_mgr.load(model_restored, optimizer_restored, ckpt_path)
print(f"✅ Restored from step {restored_step}")

# Verify outputs match
test_x, _ = get_batch(val_ids, 2, seq_len)
out_orig = model(test_x)
out_rest = model_restored(test_x)
mx.eval(out_orig, out_rest)
max_diff = float(mx.max(mx.abs(out_orig - out_rest)).item())
print(f"🎯 Max output difference after restore: {max_diff:.2e} (should be ~0)")

# NaN detection demo
print(f"\n⚠️ NaN detection: {ckpt_mgr.detect_nan(float('nan'))} (True = would trigger recovery)")
print(f"   Normal loss:   {ckpt_mgr.detect_nan(2.5)} (False = all good)")

💾 Saved checkpoint to: checkpoints/gpt_demo/ckpt_step_200
✅ Restored from step 200
🎯 Max output difference after restore: 0.00e+00 (should be ~0)

⚠️ NaN detection: True (True = would trigger recovery)
   Normal loss:   False (False = all good)


## 📊 Task 9.6 — Evaluation & Generation Samples

🎯 **Perplexity** = exp(cross-entropy loss). A perplexity of *P* means the model is as uncertain as choosing uniformly among *P* options at each step. Lower is better.

We show generation samples to visualize quality progression.

In [16]:
from utils.training import evaluate, generate_text

# Evaluate on validation set
val_loss, ppl = evaluate(model, val_ids, batch_size=8, seq_len=seq_len, n_batches=20)
print(f"📊 Validation Loss: {val_loss:.4f}  |  Perplexity: {ppl:.1f}")

# Plot training loss + gradient norms
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

axes[0].plot(train_losses, alpha=0.7)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss'); axes[0].grid(True, alpha=0.3)

axes[1].plot(grad_norms, alpha=0.7, color='orange')
axes[1].axhline(clip_norm, color='red', linestyle='--', alpha=0.5, label=f'clip={clip_norm}')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Gradient Norm')
axes[1].set_title('Gradient Norms'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(lr_history, alpha=0.7, color='green')
axes[2].set_xlabel('Step'); axes[2].set_ylabel('LR')
axes[2].set_title('Learning Rate Schedule'); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

📊 Validation Loss: 2.5911  |  Perplexity: 13.3


/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_24984/1372543673.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


The next cell continues the implementation:

**Generation samples — show quality progression**

In [17]:
# Generation samples — show quality progression
prompts = ["ROMEO:", "To be", "The king"]
print("=" * 60)
print("✍️  Generation Samples (after training)")
print("=" * 60)
for prompt in prompts:
    generated = generate_text(model, tokenizer, prompt, max_tokens=150, temperature=0.8)
    print(f"\nPrompt: \"{prompt}\"")
    print(f"Generated: \"{generated[:200]}\"")
    print("-" * 60)

✍️  Generation Samples (after training)

Prompt: "ROMEO:"
Generated: "ROMEO:
TA:
qZ, e l ovit'llavhereThe m buresr, is wic theris we acut f har pthat lly mal be hlld the s cad u pd.
Miterd maneserondor.
ARD hitot we:
OThThe to"
------------------------------------------------------------



Prompt: "To be"
Generated: "To beorear a alder l nd EDAr I yenoousheithind med se stendivav y w akolobrllis hes thea s'dit m fu s ar! be,
Gel's brsemaneave.
ONLD


AUO
IUTI-NAR:
AHer."
------------------------------------------------------------



Prompt: "The king"
Generated: "The kinge?esore ant bin te thof
OHes bumo apancoud pe me shoue he he, lls

Youthve sst artour b:
Thave wee ow theluifo t lalle She no t dindCisthe ly ma he s "
------------------------------------------------------------


---

### 🎯 Interview Question nb07-q03  ·  [core]  ·  mle, systems_engineer

**Q:** Derive the KV-cache memory formula `2 · L · T · n_kv_heads · d_head · bytes`. At LLaMA-3-8B (L=32, n_heads=32, n_kv_heads=8, d_head=128) with T=128k, bf16, single sequence, how many bytes? Why does this become the dominant memory cost at long context?

<details>
<summary>Key points in a strong answer</summary>

- Per token, per layer, the model stores two tensors: K and V. Each has shape `(n_kv_heads, d_head)` for the single new token. So per (token, layer): `2 · n_kv_heads · d_head · bytes_per_elem`.
- Across all L layers and T tokens in the cache: `2 · L · T · n_kv_heads · d_head · bytes`. The factor of 2 is K + V; GQA (n_kv_heads < n_heads) is what shrinks the formula relative to MHA.
- Plugging in LLaMA-3-8B at T=128k, bf16 (2 bytes): `2 · 32 · 131072 · 8 · 128 · 2 = 17,179,869,184` bytes = 16 GiB per single sequence. Larger than the entire 8B-param model weights (~16 GB bf16).
- Scaling is LINEAR in T but with a LARGE constant. At short context (T=2k) the cache is ~256 MiB — negligible. At T=128k it's 16 GiB — dominant. At T=1M it's 128 GiB — requires a multi-GPU setup or aggressive cache compression.
- Implication for serving: at long context, you run out of HBM before you run out of FLOPs. Production stacks handle this via PagedAttention (avoid fragmentation), prefix caching (share across requests), and quantized KV (int8/int4 cache cuts 2-4× more).
- Without GQA (MHA: n_kv_heads=n_heads): at LLaMA-3-8B, cache grows to `2 · 32 · 131072 · 32 · 128 · 2` = 64 GiB per sequence — why GQA was the single most impactful architecture change for long-context serving in 2023-2024.
- Multi-head Latent Attention (MLA, DeepSeek-V3) projects K and V into a shared low-rank latent; cache shrinks ~10× vs GQA at comparable quality. Frontier-2024 alternative.
</details>

> ⚠️ **Trap:** Forgetting the factor of 2 (K and V are stored separately, not fused). Also forgetting that `n_kv_heads < n_heads` in GQA — candidates who use `n_heads` in the formula overestimate cache size by 4-8× on modern models.
>
> 📚 **References:** https://ai.meta.com/blog/meta-llama-3/, https://arxiv.org/abs/2305.13245, https://arxiv.org/abs/2309.06180

---

### 🎯 Interview Question nb07-q04  ·  [core]  ·  mle, research_engineer

**Q:** Compare three repetition-control mechanisms: n-gram blocking, HuggingFace-style `repetition_penalty`, and frequency/presence penalty (OpenAI). Write the math for each, and explain when each fails.

<details>
<summary>Key points in a strong answer</summary>

- **n-gram blocking** (`no_repeat_ngram_size=k`): before sampling, force `logit[t] = -∞` for any token that would complete an n-gram of size `k` that already appeared in the generated prefix. Hard constraint. Fails on legitimate repetition (formal poetry, code with common patterns, JSON key repetition).
- **Repetition penalty** (Keskar et al. 2019 / HuggingFace): `logit[t] = logit[t] / penalty` if `logit[t] > 0` else `logit[t] * penalty` for any token `t` already in the prefix. Soft multiplicative penalty, typically 1.0–1.3. Applied BEFORE softmax. Fails because it penalizes ALL prior tokens equally — whether they appeared once 100 steps ago or 50 times in the last 20 steps.
- **Frequency penalty + presence penalty** (OpenAI API): `logit[t] = logit[t] - frequency_penalty · count(t) - presence_penalty · indicator(t appears at all)`. Decouples "how often" (frequency) from "has it appeared at all" (presence). Additive. More precise control. Default values are 0 (off).
- The three fail modes they leave unsolved: (a) SEMANTIC repetition (model paraphrases itself — n-gram blocking doesn't catch it); (b) POSITIONAL repetition (model repeats a word every k tokens — penalties decay over long contexts); (c) HIGH-ENTROPY LOOPS (model cycles through 3-4 tokens — penalties don't break the cycle if all four are below the penalty threshold).
- Production stacks combine approaches: vLLM + SGLang expose all three via sampling params; typical default is `presence_penalty=1.0, frequency_penalty=0.2` for chat.
- Better alternatives emerging (2024): DoLa (contrastive decoding between early-layer and final-layer logits), Mirostat (adaptive sampling targeting a perplexity set-point), min-p (sample from tokens whose probability is ≥ p · max_prob — a relative threshold that adapts to distribution sharpness).
</details>

> ⚠️ **Trap:** Applying `repetition_penalty` AFTER softmax instead of to the raw logits. Post-softmax division of probabilities is mathematically different and breaks the probability simplex. Every correct implementation modifies logits BEFORE softmax.
>
> 📚 **References:** https://arxiv.org/abs/1909.05858, https://platform.openai.com/docs/guides/text-generation, https://arxiv.org/abs/2309.03883

---

### 🎯 Interview Question nb07-q05  ·  [stretch]  ·  research_engineer, systems_engineer

**Q:** Why is prefill compute-bound (O(B·T²·d)) but decode memory-bandwidth-bound (O(L·T·n_kv_heads·d_head))? What does this imply for batching strategy?

<details>
<summary>Key points in a strong answer</summary>

- **Prefill** (processing the prompt): all T prompt tokens pass through the model in a single forward pass. Attention has O(T²) compute per layer; matmul ops saturate the tensor cores. FLOPs ≈ 2 · T · P (P = model params) per token, times T prompt tokens = 2·T²·P total. High arithmetic intensity — WELL above the H100 ridge point (~1000 FLOPs/byte). Compute-bound.
- **Decode** (generating the N+1st token): only ONE query token attends to the full T-token KV cache. Per step: read the ENTIRE KV cache (`2·L·T·n_kv_heads·d_head·bytes`) from HBM, do one `(1, d) × (T, d)` attention matmul. Arithmetic intensity ≈ d/2 (same as attention; see NB05-q07) — BELOW the ridge point. Memory-bandwidth-bound.
- Numerical illustration at LLaMA-3-70B, T=8k, bf16: prefill requires ~560 TFLOPs for one sequence (takes ~1s on H100); decode requires ~40 GiB/step of HBM reads (takes ~40ms/step at 1 TB/s bandwidth) — the hardware is IDLE on compute during decode.
- **Batching implications**: decode is memory-bandwidth-bound → stacking more sequences DOES NOT increase per-step latency (same KV reads, just more queries). Continuous batching (vLLM) exploits this: pack 64+ sequences and each gets nearly the same TPOT. Prefill is compute-bound → batching LINEARLY increases TTFT per request (shared compute). Optimal: prefer LARGE batches for decode, SMALL batches (chunked) for prefill.
- **Chunked prefill** (vLLM v1 default in 2024): break long prompts into chunks, interleave prefill chunks with decode steps from the pool. Keeps latency low for ongoing generations while new prompts are filled in. The only way to serve long-context at high throughput.
- Implication for hardware choice: decode-dominated workloads (chat with short prompts, long responses) are bandwidth-bound → Apple Silicon M4 Max (546 GB/s) hits ~40% of H100 (3 TB/s) per sequence, but costs ~1/50 the power. Prefill-dominated workloads (RAG with long context, short responses) are compute-bound → H100 dominates because it has ~20× more bf16 FLOPs.
- Speculative decoding (Leviathan et al. 2023, Medusa 2024) changes the equation: a small draft model proposes multiple tokens per step, target model verifies in parallel — effectively one forward pass generates ~4-8 accepted tokens. Turns decode from memory-bound to PARTIALLY compute-bound.
</details>

> ⚠️ **Trap:** Saying "decode is also O(T²) because attention is O(T²)". Per-step decode is ONLY O(T) — the single new query attends to T cached keys. Cumulative over T generated tokens it's O(T²), but any one step is linear in T. This distinction is what makes streaming possible.
>
> 📚 **References:** https://arxiv.org/abs/2309.06180, https://arxiv.org/abs/2211.17192, https://docs.vllm.ai/en/latest/design/v1/v1_design.html

---

### 🎯 Interview Question nb07-q06  ·  [stretch]  ·  mle, systems_engineer

**Q:** What is TTFT (time-to-first-token) and why does it explode at long context? Explain chunked prefill, prefix caching, and why a RAG system with a 32k-token system prompt needs both.

<details>
<summary>Key points in a strong answer</summary>

- **TTFT** (time-to-first-token): wall-clock latency from request arrival to the FIRST output token. Dominated by PREFILL cost — the forward pass on the prompt. Unlike TPOT (time-per-output-token), TTFT is quadratic in prompt length: ~2·T²·P/FLOPs_per_sec at large T.
- Numerical floor at LLaMA-3-70B: TTFT at T=8k ≈ 1-2 seconds on H100; at T=32k ≈ 16-32 seconds; at T=128k ≈ 2-5 MINUTES. At frontier context sizes, pure prefill is a user-visible pause.
- **Chunked prefill** (vLLM v1, SGLang): split the prompt into chunks of C tokens (e.g., C=512) and interleave prefill chunks with decode steps from the ongoing pool. The scheduler runs one chunk per step, so TTFT for the new request starts streaming after `ceil(T/C)` chunks — O(T/C) latency instead of O(T²), at the cost of slightly slower decode on the concurrent sequences.
- **Prefix caching** (vLLM's `enable_prefix_caching`, SGLang's RadixAttention): persist the KV cache for a frequently-reused prefix (a system prompt, a few-shot template, a shared document). Subsequent requests with the same prefix skip prefill for those tokens entirely — TTFT drops to the cost of the DELTA tokens (often <1s regardless of prefix length).
- Why RAG with 32k system prompt needs BOTH: (a) prefix caching: the 32k context is typically reused across many retrieval queries — cache the prefill so the per-query cost is just the query tokens; (b) chunked prefill: the first time the system prompt is seen (cache miss), the model still needs to prefill it — chunked prefill keeps the server responsive to OTHER requests during that minutes-long prefill.
- Cache-hit rate is the #1 operational metric for RAG serving. Typical deployments: 95%+ hit rate on system-prompt prefix, 20-40% hit rate on document-chunk prefixes. A 90%→99% hit rate improvement often doubles effective throughput more than hardware upgrades.
- Prefix cache correctness: cache is keyed by the HASH of the token sequence (tokens as ints, not text — so tokenizer changes invalidate the cache). Position-encoding IDs must start from 0 for the prefix but continue from `len(prefix)` for the new tokens — off-by-one here causes the classic "prefix cache silently wrong" bug.
</details>

> ⚠️ **Trap:** Conflating TTFT with TPOT. They're orthogonal metrics with different bottlenecks. TTFT optimization is about prefill (chunking, caching, batching SMALLER). TPOT optimization is about decode (continuous batching, KV compression, speculative decoding). A system can have great TTFT and bad TPOT or vice versa.
>
> 📚 **References:** https://docs.vllm.ai/en/latest/features/automatic_prefix_caching.html, https://lmsys.org/blog/2024-01-17-sglang/, https://arxiv.org/abs/2308.16369

---

### 🎯 Interview Question nb07-q07  ·  [research]  ·  research_engineer, systems_engineer

**Q:** Explain the 2024–2026 frontier on decode-time acceleration: speculative decoding, Medusa, EAGLE-2, Kangaroo early-exit, and vLLM v1. What's the common theme and what's the open problem?

<details>
<summary>Key points in a strong answer</summary>

- **Core insight**: decode is memory-bandwidth-bound. One forward pass at batch=1 reads the entire KV cache to produce ONE token. If you can somehow generate MULTIPLE tokens per forward pass without re-reading the cache, throughput multiplies. This is the unifying theme.
- **Speculative decoding** (Leviathan, Chen et al. 2023): a small DRAFT model (e.g., LLaMA-3-1B) proposes `k` tokens greedily in a cheap forward pass; the TARGET model (70B) verifies all `k` in ONE forward pass. Accept `m ≤ k` tokens that agree with target's greedy choice. Speedup = k · acceptance_rate / (cost_draft + 1). Typical: 2-3× on chat workloads with LLaMA-3-8B drafter + 70B target.
- **Medusa** (2024): train ADDITIONAL decoder heads on the base model itself — each predicts token t+1, t+2, t+3 via its own projection layer. No separate draft model needed. Heads are cheap (~0.2% of model params). Uses tree-structured verification to accept the longest valid path. 2-3× speedup with 0.5% training overhead.
- **EAGLE / EAGLE-2** (2024): the most precise variant — trains a small draft that operates on the base model's LAST-LAYER HIDDEN STATES (not tokens). Draft predicts next hidden state, then the target's LM head verifies. Achieves higher acceptance rate than token-level drafts because hidden states carry more information. Reported 2.5-3.5× on production chat.
- **Kangaroo** (2024): EARLY-EXIT-based — during the target's forward pass, periodically consult a smaller "exit" head. If confident, emit that token and skip the remaining layers for this position. No draft model; just dynamic compute allocation. Harder to get right but lower memory overhead.
- **vLLM v1** (2025): production packaging of all of these. Chunked prefill + continuous batching + speculative decoding + prefix caching in one scheduler. The "decode-time compute multiplier" stack of 2025: 2-4× from spec-decode × 1.3× from prefix cache × 1.5× from chunked prefill = 4-8× throughput vs naive 2023 serving.
- **Open problem**: all of these speedup methods assume MOSTLY-DETERMINISTIC output. As soon as temperature > 0.7 or top-p < 0.9, acceptance rates drop below 50% and speedups collapse. Reasoning models (o1, DeepSeek-R1) with long high-entropy CoT chains are hardest to accelerate. 2025-2026 research frontier: spec-decode variants tuned for high-entropy sampling.
- **Correctness invariant** across all spec-decode variants: the OUTPUT DISTRIBUTION must exactly match greedy (for greedy mode) or the target's sampled distribution (for temperature > 0). Leviathan 2023 proves this with an explicit rejection-sampling construction. A spec-decode implementation that changes the output distribution is BROKEN — this catches 90% of buggy implementations.
</details>

> ⚠️ **Trap:** Thinking spec-decode trades quality for speed. It's LOSSLESS for the target distribution — the output is provably identical to running the target model alone. Only the LATENCY changes. Interviewers probe this by asking "if I have a 30% acceptance rate, does output quality degrade?" Answer: no, only the speedup drops.
>
> 📚 **References:** https://arxiv.org/abs/2211.17192, https://arxiv.org/abs/2401.10774, https://arxiv.org/abs/2406.16858, https://docs.vllm.ai/en/latest/design/v1/v1_design.html

---

### 🧑‍💻 Whiteboard Challenge: Implement a simple KV-cache and verify its memory footprint

**Prompt:** Implement a `KVCache` class that stores `(k, v)` tensors per layer and an `append(new_k, new_v)` method that grows the cache along the T axis. Assert (a) the `bytes_used()` method returns a value within 1% of the analytic `2 · L · T · n_kv_heads · d_head · bytes` formula, and (b) the cache grows LINEARLY in the number of append calls.

**Constraints:**
- Use MLX arrays throughout. Store per-layer `k` and `v` as `mx.array` of shape `(B, n_kv_heads, T, d_head)`.
- `append` should use `mx.concatenate(..., axis=2)` to extend along the sequence axis.
- Return the analytic formula from a `bytes_analytic()` classmethod and the measured bytes from `bytes_measured()` (sum of `k.nbytes + v.nbytes` across layers).
- `assert abs(measured - analytic) / analytic < 0.01` for a few `(L, T, n_kv_heads, d_head)` combinations.
- Call `mx.eval` before measuring bytes so shapes/allocations are realized.

**Expected complexity:** O(L · T · n_kv_heads · d_head) memory per sequence; `append` is O(L · (T_new - T_old) · n_kv_heads · d_head) to materialize the new tensor. Modern production stacks (PagedAttention) avoid the `concatenate` cost via block-paged storage — but this toy version keeps the formula clean.

In [18]:
import mlx.core as mx


class _KVCache:
    """Minimal per-layer KV cache. Each layer stores (k, v) of shape (B, n_kv_heads, T, d_head)."""

    def __init__(self, n_layers: int, batch_size: int, n_kv_heads: int, d_head: int,
                 dtype=mx.bfloat16):
        self.n_layers = n_layers
        self.B = batch_size
        self.n_kv_heads = n_kv_heads
        self.d_head = d_head
        self.dtype = dtype
        # Start with zero-length cache per layer.
        self._k = [mx.zeros((batch_size, n_kv_heads, 0, d_head), dtype=dtype)
                   for _ in range(n_layers)]
        self._v = [mx.zeros((batch_size, n_kv_heads, 0, d_head), dtype=dtype)
                   for _ in range(n_layers)]

    def append(self, layer_idx: int, new_k: mx.array, new_v: mx.array) -> None:
        """Extend layer `layer_idx`'s cache by T_new tokens. new_k / new_v shape: (B, n_kv_heads, T_new, d_head)."""
        self._k[layer_idx] = mx.concatenate([self._k[layer_idx], new_k], axis=2)
        self._v[layer_idx] = mx.concatenate([self._v[layer_idx], new_v], axis=2)

    @property
    def T(self) -> int:
        """Current cached sequence length (assumed identical across layers)."""
        return int(self._k[0].shape[2])

    def bytes_measured(self) -> int:
        """Sum nbytes across K and V across all layers."""
        mx.eval(*self._k, *self._v)
        return sum(k.nbytes + v.nbytes for k, v in zip(self._k, self._v))

    @classmethod
    def bytes_analytic(cls, n_layers: int, batch_size: int, T: int,
                       n_kv_heads: int, d_head: int, dtype_bytes: int = 2) -> int:
        """Return 2 · L · B · T · n_kv_heads · d_head · bytes."""
        return 2 * n_layers * batch_size * T * n_kv_heads * d_head * dtype_bytes


# Build a LLaMA-3-8B-scale cache and verify the formula at several T values.
_L, _B, _n_kv, _d_head = 32, 1, 8, 128
_cache = _KVCache(_L, _B, _n_kv, _d_head, dtype=mx.bfloat16)

# Append increments of 128 tokens at a time, checking the memory formula after each append.
_T_points = [128, 512, 2048, 8192]
_prev_T = 0
_ratios = []
for _T in _T_points:
    _delta = _T - _prev_T
    # Fresh random (new_k, new_v) of shape (B, n_kv_heads, delta, d_head) per layer.
    mx.random.seed(0)
    for _layer in range(_L):
        _new_k = mx.random.normal(shape=(_B, _n_kv, _delta, _d_head)).astype(mx.bfloat16)
        _new_v = mx.random.normal(shape=(_B, _n_kv, _delta, _d_head)).astype(mx.bfloat16)
        _cache.append(_layer, _new_k, _new_v)
    _prev_T = _T

    _analytic = _KVCache.bytes_analytic(_L, _B, _T, _n_kv, _d_head, dtype_bytes=2)
    _measured = _cache.bytes_measured()
    _ratio = _measured / _analytic
    _ratios.append(_ratio)
    print(f"T={_T:>5} | analytic = {_analytic / 1e6:>7.2f} MB | measured = {_measured / 1e6:>7.2f} MB | ratio = {_ratio:.4f}")

# Primary assertion: measured matches analytic to within 1% at every checkpoint.
for _T, _r in zip(_T_points, _ratios):
    assert abs(_r - 1.0) < 0.01, f"T={_T}: measured/analytic = {_r:.4f}, expected within 1%"

# Secondary assertion: current cache length equals the final append target.
assert _cache.T == 8192, f"cache length mismatch: {_cache.T}"

# At T=128k the cache would need 16 GiB of bf16 — enormous.
# We don't actually allocate it here; we just compute the projection.
_projected = _KVCache.bytes_analytic(_L, _B, 131_072, _n_kv, _d_head, dtype_bytes=2)
print(f"\nProjected bytes at T=128k (LLaMA-3-8B single seq, bf16): {_projected / (1024**3):.2f} GiB")
print(f"✅ Formula validated across {len(_T_points)} cache sizes (all within 1% of analytic).")

T=  128 | analytic =   16.78 MB | measured =   16.78 MB | ratio = 1.0000
T=  512 | analytic =   67.11 MB | measured =   67.11 MB | ratio = 1.0000
T= 2048 | analytic =  268.44 MB | measured =  268.44 MB | ratio = 1.0000


T= 8192 | analytic = 1073.74 MB | measured = 1073.74 MB | ratio = 1.0000

Projected bytes at T=128k (LLaMA-3-8B single seq, bf16): 16.00 GiB
✅ Formula validated across 4 cache sizes (all within 1% of analytic).


---

### 📐 Complexity & Systems: Prefill vs Decode throughput

| Quantity | Formula / Value | Notes |
|---|---|---|
| Prefill FLOPs | `~2 · T · P` per token, × T tokens = `2 · T² · P` total. At T=2048, LLaMA-3-8B: ~67 TFLOPs for one prompt forward | compute-bound |
| Decode FLOPs | `~2 · P` per generated token + attention `O(L · T · n_kv_heads · d_head)` per step. At T=2048: ~16 GFLOPs/step | bandwidth-bound at batch 1 |
| Decode HBM reads | `2 · L · T · n_kv_heads · d_head · bytes` per step — scales LINEARLY with cache length | dominant cost |
| Latency (M4 Pro, MLX) | Measured below at (d_model=128, n_layers=2, T ∈ {128, 512, 2048}) — small toy model; relative ratios hold | see benchmark |

💡 **Scaling implication:** Prefill is quadratic in T; decode is linear per step but every step reads the full cache. Doubling batch size at prefill ~doubles prefill time (compute-bound); doubling batch size at decode is ~free (bandwidth-bound amortizes). This asymmetry is what continuous batching exploits.

In [19]:
import time
import mlx.core as mx
import mlx.nn as _nn


# Tiny model for illustration; the RATIOS hold across model sizes.
# Names prefixed with underscore to avoid colliding with earlier cells.
_vocab = 128
_d = 128
_n_heads = 4
_n_layers = 2
_d_ff = 512

class _Tiny(_nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = _nn.Embedding(_vocab, _d)
        self.blocks = [_nn.TransformerEncoderLayer(dims=_d, num_heads=_n_heads,
                                                   mlp_dims=_d_ff, dropout=0.0)
                       for _ in range(_n_layers)]
        self.lm = _nn.Linear(_d, _vocab, bias=False)

    def __call__(self, toks: mx.array) -> mx.array:
        x = self.emb(toks)
        for b in self.blocks:
            x = b(x, mask=None)
        return self.lm(x)

_model = _Tiny()
mx.eval(_model.parameters())

def _prefill(T: int) -> float:
    """Time a single forward pass on a prompt of length T."""
    mx.random.seed(0)
    _toks = mx.random.randint(0, _vocab, shape=(1, T))
    mx.eval(_toks)
    for _ in range(3):
        _ = _model(_toks); mx.eval(_)
    _t0 = time.perf_counter()
    for _ in range(5):
        _ = _model(_toks); mx.eval(_)
    return (time.perf_counter() - _t0) / 5 * 1000.0

def _decode_step(T_cache: int) -> float:
    """Time one decode step: model sees 1 new token with T_cache-length history."""
    mx.random.seed(0)
    _toks = mx.random.randint(0, _vocab, shape=(1, T_cache + 1))
    mx.eval(_toks)
    for _ in range(3):
        _ = _model(_toks); mx.eval(_)
    _t0 = time.perf_counter()
    for _ in range(5):
        _ = _model(_toks); mx.eval(_)
    return (time.perf_counter() - _t0) / 5 * 1000.0

print(f"{'T':>6} | {'prefill ms':>12} | {'decode step ms':>16} | {'tokens/sec (decode)':>22}")
print("-" * 64)
for _T in (128, 512, 2048):
    _pfill = _prefill(_T)
    _dec = _decode_step(_T)
    _tps = 1000.0 / _dec if _dec > 0 else 0.0
    print(f"{_T:>6} | {_pfill:>10.2f} ms | {_dec:>14.2f} ms | {_tps:>20.1f}")

# Sanity: at this toy scale, prefill cost grows ~quadratically with T
# (attention is T²); decode cost grows ~linearly with T (cache lookup).
# We don't assert exact ratios because the M4 Pro scheduler variance is ~10-20%
# at these small sizes.
print("\n💡 Ratios to remember: prefill grows as T², decode per-step grows as T.")
print("   Continuous batching amortizes decode HBM reads across concurrent sequences.")

     T |   prefill ms |   decode step ms |    tokens/sec (decode)
----------------------------------------------------------------
   128 |       0.48 ms |           0.46 ms |               2197.0
   512 |       0.83 ms |           0.90 ms |               1107.6


  2048 |       4.02 ms |           4.30 ms |                232.4

💡 Ratios to remember: prefill grows as T², decode per-step grows as T.
   Continuous batching amortizes decode HBM reads across concurrent sequences.


---

### 🏭 How Production Systems Handle This — KV-cache management & continuous batching

| System | Mechanism | Notes |
|---|---|---|
| vLLM | PagedAttention: physical KV memory split into fixed-size BLOCKS (16 tokens each); logical sequence → list of block indices via a page table. Eliminates internal fragmentation; enables prefix sharing (two requests pointing to same blocks). v1 adds chunked prefill. | |
| SGLang | RadixAttention: reuses the KV cache across requests whose prompts share a prefix (system prompt, few-shot examples). Prefix is a trie; cache-hit means the shared prefix is NEVER re-prefilled. Integrates with speculative decoding for additional 2-3× decode speedup. | |
| TensorRT-LLM | In-flight batching (continuous batching): sequences added/removed from the running batch every step; no explicit prefill/decode split. Requires pre-compiled engines per (max_seq_len, batch_size). FP8 KV cache support on H100/H200 saves 2× memory. | |
| MLX-LM | Rotating KV cache: fixed-size ring buffer for each layer; when T exceeds `max_kv_size`, the OLDEST entries are overwritten (sliding-window style). Simple, UMA-friendly. Less sophisticated than PagedAttention but correct for single-request streaming. | |

🎯 **Interview tip:** Know at least one concrete trade-off per row.

---

### 🔭 Frontier Context (Decode-time acceleration, 2024–2026)

**Paper trail:**
1. Speculative Decoding (Leviathan, Chen et al., 2023) — Small draft model proposes k tokens; target verifies in one pass. Lossless for greedy; rejection-sampling extends to temperature > 0. Typical 2-3× speedup with 8B drafter + 70B target on chat workloads.
2. Medusa: Simple Framework for Accelerating LLM Generation with Multiple Decoding Heads (Cai et al., 2024) — Adds k extra prediction heads to the target model itself; no separate draft. Tree-verification accepts the longest valid path. 2-3× at ~0.5% training overhead.
3. EAGLE / EAGLE-2 (Li et al., 2024) — Draft operates on target's LAST-LAYER HIDDEN STATES, not tokens. Higher acceptance rate (~80%) than token-level drafts. Production default in SGLang's spec-decode in 2024.
4. Kangaroo: Lossless Self-Speculative Decoding via Double Early Exiting (Liu et al., 2024) — Early-exit-based: target model periodically consults an "exit" head during its forward pass. No separate draft; just dynamic compute. Best fit when you can't afford a second model.
5. vLLM v1 (2025 redesign) — Packages chunked prefill + continuous batching + prefix caching + spec-decode into one scheduler. Cumulative 4-8× throughput vs 2023-era naive serving. Current production default for LLaMA-3 / Qwen / DeepSeek-V3 deployments.

**Current SOTA:** Late-2025 production stack = pre-norm + RMSNorm + GQA/MLA + FA3 + PagedAttention/RadixAttention + chunked prefill + EAGLE-2 spec-decode + prefix caching + FP8 KV. Active frontiers: (i) accelerating high-entropy sampling (reasoning models, o1-style), (ii) sub-quadratic attention alternatives (NSA, MoBA 2025) that bypass the KV cache problem entirely, (iii) hardware-aware scheduling (vLLM v2 previews predict batch composition from GPU utilization signals).

---

### 🛠️ Failure Modes & Debugging: Three generation-time bugs

**Root causes (ranked by frequency):**
1. **KV-cache stale after batch eviction**: a sequence finishes and its slot is reused for a new request, but the serving stack forgot to RESET the KV-cache pointers for that slot. Symptom: the new request's first few tokens are contaminated by the old sequence's hidden state — generation starts with garbage or a persona bleed. Fix: on slot reuse, zero the (K, V) buffers for that slot (or better, maintain a logical `seq_id` and refuse to reuse the slot without a fresh allocation).
2. **Off-by-one in RoPE position ids when appending to KV cache**: generated token at step `t` must use position `past_len + t`, NOT `t` alone (the KV cache already holds `past_len` entries). Bug: using `t` makes every generated token "see itself as position 0" — the model thinks every generation restarts from scratch and outputs are incoherent after the first few tokens. Fix: the attention module must receive `position_ids = past_len + arange(T_new)` explicitly.
3. **Repetition loop without penalty**: pure greedy sampling at temperature=0 + no repetition penalty on a short-context model reliably produces "The the the the..." loops because the model's argmax distribution peaks on the same token conditional on the recent few tokens. Fix: set `repetition_penalty >= 1.1` OR use top-p ≥ 0.9 with temperature ≥ 0.7. The loop is a property of the TRAINING distribution, not the sampler — but a non-trivial sampler breaks it.

**Diagnostic code below reproduces the symptom then shows the fix:**

In [20]:
import mlx.core as mx


# -- Symptom 1: stale KV cache on slot reuse --------------------
# Simulate a tiny KV "buffer" shared between two sequences.
# If slot reuse doesn't zero out old state, the new sequence inherits it.
_B, _T_cache, _d = 1, 8, 16
mx.random.seed(0)
_old_k = mx.random.normal(shape=(_B, _T_cache, _d))
mx.eval(_old_k)

# Bug: reuse the slot without zeroing — new K starts "contaminated".
_reused_buggy = _old_k  # slot pointer carries old data

# Fix: explicitly reset the slot before assigning the new sequence's K.
_reset_fix = mx.zeros((_B, _T_cache, _d), dtype=_old_k.dtype)

# The diagnostic signal is the mean magnitude — it should be ~0 after reset.
_buggy_mag = float(mx.mean(mx.abs(_reused_buggy)).item())
_fixed_mag = float(mx.mean(mx.abs(_reset_fix)).item())
print(f"[1] stale-cache bug:  |old K| retained = {_buggy_mag:.4f}")
print(f"    fix (zero reset): |new K| = {_fixed_mag:.4f}")
assert _buggy_mag > 0.1, "buggy case should carry non-trivial old-K magnitude"
assert _fixed_mag < 1e-6, "fix should zero out the slot"
print("    → symptom: new request sees previous request's hidden-state context.\n")


# -- Symptom 2: off-by-one in RoPE position ids -----------------
# Correct: position_ids = past_len + arange(T_new).
# Bug: just arange(T_new), treating every generation restart at position 0.
_past_len = 20
_T_new = 4
_correct_pos = _past_len + mx.arange(_T_new)
_buggy_pos = mx.arange(_T_new)
mx.eval(_correct_pos, _buggy_pos)

# The two position tensors must disagree on every entry.
_diff = int(mx.sum((_correct_pos != _buggy_pos).astype(mx.int32)).item())
assert _diff == _T_new, "correct and buggy position tensors must all disagree"
print(f"[2] position ids:")
print(f"    correct (past_len + arange): {_correct_pos.tolist()}")
print(f"    buggy (arange only):         {_buggy_pos.tolist()}")
print(f"    → symptom: after token 20, model treats every generated token as 'position 0' — outputs incoherent.\n")


# -- Symptom 3: greedy without repetition penalty loops --------
# Simulate a logit vector where token 0 is always the greedy argmax,
# and show how applying a repetition penalty breaks that stable point.
_V = 8
_logits = mx.array([2.0, 1.0, 0.5, 0.0, -0.1, -0.2, -0.3, -0.4])
_history = [0, 0, 0]  # previous generations all chose token 0

# Greedy: always picks token 0, forever.
_greedy_next = int(mx.argmax(_logits).item())
assert _greedy_next == 0, f"greedy on these logits must pick 0, got {_greedy_next}"

# Apply repetition penalty (multiplicative on logits BEFORE softmax).
# For each token t in history, divide positive logit by penalty (or multiply if negative).
# We use penalty=2.5 here (higher than typical 1.1–1.3) to make the effect demonstrable
# on this small logit gap; production chat defaults are lower but operate over larger
# logit gaps and longer histories.
_penalty = 2.5
_logits_penalized = mx.array(_logits.tolist())  # copy
for _tok in set(_history):
    _v = float(_logits_penalized[_tok].item())
    if _v > 0:
        _new_v = _v / _penalty
    else:
        _new_v = _v * _penalty
    _logits_penalized = mx.concatenate([
        _logits_penalized[:_tok],
        mx.array([_new_v]),
        _logits_penalized[_tok + 1:],
    ])

mx.eval(_logits_penalized)
_penalized_next = int(mx.argmax(_logits_penalized).item())
print(f"[3] greedy without penalty: picks token {_greedy_next} (will loop forever)")
print(f"    with penalty={_penalty}:    picks token {_penalized_next} (breaks the loop)")
assert _penalized_next != _greedy_next, "penalty must shift argmax away from the repeated token"
print("    → fix: repetition_penalty >= 1.1 OR temperature > 0 with top-p sampling.")

[1] stale-cache bug:  |old K| retained = 0.8790
    fix (zero reset): |new K| = 0.0000
    → symptom: new request sees previous request's hidden-state context.

[2] position ids:
    correct (past_len + arange): [20, 21, 22, 23]
    buggy (arange only):         [0, 1, 2, 3]
    → symptom: after token 20, model treats every generated token as 'position 0' — outputs incoherent.

[3] greedy without penalty: picks token 0 (will loop forever)
    with penalty=2.5:    picks token 1 (breaks the loop)
    → fix: repetition_penalty >= 1.1 OR temperature > 0 with top-p sampling.


---

## 🧪 Try It Yourself

Experiment with your GPT model:

1. **Temperature**: Generate text with temperature=0.1, 0.5, 1.0, and 2.0. How does the output change? Why?

2. **Model size**: Try doubling d_model from 64 to 128. Does the loss decrease faster? How much more memory does it use?

3. **Training data**: Replace the training text with something different (a poem, code, etc.). How does the generated output change after training?

## 📜 History & Alternatives: The GPT Family

The GPT lineage traces the evolution of decoder-only transformers from research curiosity to the most capable AI systems:

| Year | Model | Team | Key Contribution |
|------|-------|------|-----------------|
| **2018** | **GPT-1** | OpenAI (Radford et al.) | Proved unsupervised pre-training + supervised fine-tuning works. 117M params, BooksCorpus. First to show a single pre-trained model transfers to many tasks. |
| **2019** | **GPT-2** | OpenAI (Radford et al.) | Scaled to 1.5B params, WebText dataset. Zero-shot task performance without fine-tuning. "Too dangerous to release" controversy sparked AI safety debate. |
| **2020** | **GPT-3** | OpenAI (Brown et al.) | 175B params, in-context learning via few-shot prompting. Showed scaling unlocks emergent abilities. Introduced the API-as-a-product model. |
| **2022** | **InstructGPT** | OpenAI (Ouyang et al.) | RLHF alignment — made GPT-3 follow instructions. Bridge from "predict next token" to "helpful assistant." |
| **2023** | **GPT-4** | OpenAI | Multimodal (text + vision), dramatically improved reasoning. Mixture-of-Experts architecture (rumored). Set new SOTA on professional exams. |
| **2024** | **GPT-4o** | OpenAI | Omni-modal: native audio, vision, and text in one model. Real-time voice conversation. |

### 🎯 Interview Insight: Why Decoder-Only Won

Early transformer work explored encoder-only (BERT), encoder-decoder (T5), and decoder-only (GPT) architectures. Decoder-only won for generation because:

1. **Simpler architecture** — one stack, one attention pattern (causal)
2. **Scales better** — all parameters contribute to generation
3. **Unified interface** — every task becomes "complete this text"
4. **KV-cache friendly** — causal attention enables efficient autoregressive decoding

### Alternatives & Parallel Tracks

- **BERT (2018)** — Encoder-only, bidirectional. Dominated NLU benchmarks but can't generate.
- **T5 (2019)** — Encoder-decoder, "text-to-text" framing. Still used for translation/summarization.
- **LLaMA (2023, Meta)** — Open-weight GPT-style models. Sparked the open-source LLM revolution.
- **Mistral/Mixtral (2023-24)** — Efficient architectures with sliding window attention and MoE.
- **Gemma (2024-25, Google)** — Compact, open models. Gemma 4 uses MoE + shared experts.

⚡ **The trend:** Each generation scales compute ~10-100×, but the core decoder-only transformer architecture from GPT-1 remains remarkably unchanged.

## Summary

We built a complete GPT model from scratch in MLX and trained it on Shakespeare with a production-grade pipeline:

- **Dataset**: tiny_shakespeare with 90/10 train/val split and character-level tokenization
- **LR Schedule**: Cosine decay with linear warmup (the GPT-3 recipe)
- **Gradient Clipping**: Global norm clipping to prevent training instability
- **Checkpointing**: Save/load model + optimizer state with NaN recovery
- **Evaluation**: Validation loss and perplexity tracking
- **Generation**: Temperature-controlled autoregressive sampling

**Next:** Notebook 08 — Training on Apple Silicon (memory profiling, mixed precision, gradient accumulation)

---

### 📋 Interview Question Index

| ID | Difficulty | Roles | Question |
|---|---|---|---|
| `nb07-q01` | warmup | mle, research_engineer | Derive the softmax-temperature formula `softmax(logits / T)`. What do T→0 and T→∞ limit... |
| `nb07-q02` | core | mle, research_engineer, systems_engineer | Compare greedy, top-k, top-p (nucleus), and beam search decoding. Write the math f... |
| `nb07-q03` | core | mle, systems_engineer | Derive the KV-cache memory formula `2 · L · T · n_kv_heads · d_head · bytes`. At LLaM... |
| `nb07-q04` | core | mle, research_engineer | Compare three repetition-control mechanisms: n-gram blocking, HuggingFace-style `re... |
| `nb07-q05` | stretch | research_engineer, systems_engineer | Why is prefill compute-bound (O(B·T²·d)) but decode memory-bandwidth-bound (O(L·T·n_k... |
| `nb07-q06` | stretch | mle, systems_engineer | What is TTFT (time-to-first-token) and why does it explode at long context? Explai... |
| `nb07-q07` | research | research_engineer, systems_engineer | Explain the 2024–2026 frontier on decode-time acceleration: speculative decoding, M...